In [2]:
# Fire2Air Darwin — Model 3 Count Prediction and Explainability
# Name: Navodya Piumanthi

RANDOM_STATE = 42

In [3]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_poisson_deviance
)

In [5]:
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
    print("XGBoost available")

except Exception as exc:
    HAS_XGBOOST = False
    print("XGBoost unavailable:", exc)

try:
    import shap
    HAS_SHAP = True
    print("SHAP available")

except Exception as exc:
    HAS_SHAP = False
    print("SHAP unavailable:", exc)

XGBoost available
SHAP available


In [6]:
RANDOM_STATE = 42

# Locate the Source code folder containing the shared modelling checkpoint
possible_folders = [
    Path.cwd(),
    Path.cwd() / "Source code",
    Path.cwd().parent,
    Path.cwd().parent / "Source code",
]

project_folder = None

for folder in possible_folders:
    folder = folder.resolve()

    if (
        (folder / "Fire2Air_model_ready_checkpoint.csv").exists()
        and (folder / "Fire2Air_feature_manifest.json").exists()
    ):
        project_folder = folder
        break

if project_folder is None:
    raise FileNotFoundError(
        "Could not find Fire2Air_model_ready_checkpoint.csv and "
        "Fire2Air_feature_manifest.json inside the Source code folder. "
        "Run the data-processing notebook first."
    )

print("Project folder:", project_folder)

Project folder: C:\Users\navod\Desktop\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code


In [7]:
output_root = project_folder / "outputs_prt661"
table_dir = output_root / "tables"
figure_dir = output_root / "figures"
model_dir = output_root / "models"

for folder in [table_dir, figure_dir, model_dir]:
    folder.mkdir(parents=True, exist_ok=True)

integrated_daily = pd.read_csv(
    project_folder / "Fire2Air_model_ready_checkpoint.csv",
    parse_dates=["date", "target_date"],
)

with open(
    project_folder / "Fire2Air_feature_manifest.json",
    "r",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

model_features = manifest["features"]
duration_target = manifest["targets"]["count"]

print("Project folder:", project_folder)
print("Dataset shape:", integrated_daily.shape)
print("Model 3 target:", duration_target)

Project folder: C:\Users\navod\Desktop\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code
Dataset shape: (7530, 58)
Model 3 target: target_hours_ge_25_next_day


In [8]:
duration_data = integrated_daily.dropna(
    subset=[duration_target, "target_date"]
).copy()

duration_data["target_year"] = duration_data["target_date"].dt.year

train_duration = duration_data[
    duration_data["target_year"] <= 2022
].copy()

validation_duration = duration_data[
    duration_data["target_year"] == 2023
].copy()

test_duration = duration_data[
    duration_data["target_year"] == 2024
].copy()

print("Train rows:", len(train_duration))
print("Validation rows:", len(validation_duration))
print("Test rows:", len(test_duration))

Train rows: 2415
Validation rows: 661
Test rows: 645


In [9]:
if min(
    len(train_duration),
    len(validation_duration),
    len(test_duration)
) == 0:
    raise ValueError(
        f"Insufficient chronological data: "
        f"train={len(train_duration)}, "
        f"validation={len(validation_duration)}, "
        f"test={len(test_duration)}"
    )

assert (
    train_duration["target_date"].max()
    < validation_duration["target_date"].min()
)

assert (
    validation_duration["target_date"].max()
    < test_duration["target_date"].min()
)

assert duration_data[duration_target].between(0, 24).all()

print("\nTarget distribution:")
print(
    duration_data[duration_target]
    .describe()
    .round(3)
    .to_string()
)

print(
    "Zero-hour proportion:",
    round((duration_data[duration_target] == 0).mean(), 4)
)


Target distribution:
count    3721.000
mean        2.320
std         4.243
min         0.000
25%         0.000
50%         0.000
75%         3.000
max        24.000
Zero-hour proportion: 0.6192


In [10]:
X_train = train_duration[model_features]
y_train = train_duration[duration_target]

X_validation = validation_duration[model_features]
y_validation = validation_duration[duration_target]

X_test = test_duration[model_features]
y_test = test_duration[duration_target]

print("Training features:", X_train.shape)
print("Validation features:", X_validation.shape)
print("Test features:", X_test.shape)

Training features: (2415, 22)
Validation features: (661, 22)
Test features: (645, 22)


In [11]:
categorical_features = ["station"]
numeric_features = [
    feature
    for feature in model_features
    if feature not in categorical_features
]

def onehot():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot()),
])

count_preprocessor = ColumnTransformer([
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    ),
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        numeric_features
    ),
])

tree_preprocessor = ColumnTransformer([
    (
        "categorical",
        categorical_pipeline,
        categorical_features
    ),
    (
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]),
        numeric_features
    ),
])

In [ ]:
def bounded(predictions):
    return np.clip(
        np.asarray(predictions, dtype=float),
        0,
        24
    )

def count_metrics(y_true, predictions):
    pred = bounded(predictions)

    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": float(
            np.sqrt(mean_squared_error(y_true, pred))
        ),
        "R2": r2_score(y_true, pred),
        "Mean_Poisson_Deviance": mean_poisson_deviance(
            y_true,
            np.clip(pred, 1e-6, None)
        ),
        "Prediction_Min": float(pred.min()),
        "Prediction_Max": float(pred.max()),
    }